# Mileforum — Aprendiz Episódico (v0.1)
**Timestamp:** 2026-02-04T00:30:05

Este notebook implementa el flujo **Portal → Doble Hélice → TCL → Diálogo → Metabolismo** de un *episodio*.

- Mantiene el **lenguaje interno** de cabezas (trigo/cobre/petróleo) sin renombrarlo.
- Está pensado como “**una caja con agujeros**”: ingesta mínima, ejecución, sugerencia, confirmación, registro.
- Parametrizable por rutas y por archivos JSON (`config.json` de heads y `universal_rules.json`).

> v0.1 incluye *stubs* seguros para backbone y heads si no están disponibles.

## Bloque 0 — Parámetros y Contratos
- **Entradas**: QR/NodeID + archivo (PDF/log/factura) *o* señal tabular (`INPUT_RNN_READY.csv`).
- **Estado local**: historial por nodo y learning log.
- **Salidas**: fase + claridad + artefactos de cabezas + sugerencia (TCL) + registro (DecisionEvent) + recibo de destrucción.

    • retraso en el documentado
    • cliente difícil
    • proveedor no confiable
    • presión de tiempo
    • conflicto operativo
    • rumor de huelga
    • evento externo (clima, protesta)
    • fatiga del operador
    • exceso de carga mental
    • mala comunicación

In [ ]:
import json

soft_vars_list = [
    "retraso en el documentado",
    "cliente difícil",
    "proveedor no confiable",
    "presión de tiempo",
    "conflicto operativo",
    "rumor de huelga",
    "evento externo (clima, protesta)",
    "fatiga del operador",
    "exceso de carga mental",
    "mala comunicación"
]

# Convert to the desired format for custom_soft_vars
custom_soft_vars_entries = []
for item in soft_vars_list:
    keyword = item.strip('• ').lower().replace(' ', '_').replace('(','').replace(')','').replace(',','').replace('/','_')
    custom_soft_vars_entries.append({"keyword": keyword, "dimension": "operational_risk"})

# Construct the full SOFT_STATE dictionary
initial_soft_state_content = {
    "schema_version": "1.0",
    "domain": DOMAIN["domain"],
    "custom_soft_vars": custom_soft_vars_entries
}

# Save this content to the soft_dictionary_file
soft_dictionary_path = Path(DOMAIN["soft_dictionary_file"])
soft_dictionary_path.parent.mkdir(parents=True, exist_ok=True)
soft_dictionary_path.write_text(json.dumps(initial_soft_state_content, ensure_ascii=False, indent=2), encoding="utf-8")

# Reload SOFT_STATE with the newly created file content
SOFT_STATE = load_state_dict(DOMAIN["soft_dictionary_file"], fallback=initial_soft_state_content)

print(f"✅ soft_dictionary_state.json creado/actualizado en: {soft_dictionary_path}")
print("Contenido inicial de SOFT_STATE:")
display(SOFT_STATE)

✅ soft_dictionary_state.json creado/actualizado en: /content/content/soft_dictionary_state.json
Contenido inicial de SOFT_STATE:


{'schema_version': '1.0',
 'domain': 'logistica',
 'custom_soft_vars': [{'keyword': 'retraso_en_el_documentado',
   'dimension': 'operational_risk'},
  {'keyword': 'cliente_difícil', 'dimension': 'operational_risk'},
  {'keyword': 'proveedor_no_confiable', 'dimension': 'operational_risk'},
  {'keyword': 'presión_de_tiempo', 'dimension': 'operational_risk'},
  {'keyword': 'conflicto_operativo', 'dimension': 'operational_risk'},
  {'keyword': 'rumor_de_huelga', 'dimension': 'operational_risk'},
  {'keyword': 'evento_externo_clima_protesta',
   'dimension': 'operational_risk'},
  {'keyword': 'fatiga_del_operador', 'dimension': 'operational_risk'},
  {'keyword': 'exceso_de_carga_mental', 'dimension': 'operational_risk'},
  {'keyword': 'mala_comunicación', 'dimension': 'operational_risk'}]}

In [ ]:
# === Imports (mínimos) ===
import os, io, re, time, shutil, zipfile, hashlib
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import pandas as pd

# UI
import ipywidgets as W
from IPython.display import display, Markdown, clear_output

# Torch opcional (si backbone real está en .pt)
try:
    import torch
except Exception:
    torch = None

ROOT = Path.cwd()
print("OK — imports.")

OK — imports.


In [ ]:
from pathlib import Path
ROOT = Path.cwd()

PARAMS = {
    # Artefactos
    "backbone_dir": str(ROOT / "content" / "backbone"),
    "heads_zip": str(ROOT / "logistica_multiceph_bundle_v1.zip"), # Updated path
    "universal_rules_file": str(ROOT / "content" / "universal_rules.json"),

    # Datos
    "data_dir": str(ROOT / "content"),
    "backbone_input_file": str(ROOT / "content" / "INPUT_RNN_READY.csv"),

    # Salidas / estado local
    "local_store_dir": str(ROOT / "local_store"),
    "bundle_dir": str(ROOT / "bundles_out"),
    "learning_log_file": str(ROOT / "local_store" / "learning_log.jsonl"),
}

# Dominio (unipersonal u organización)
DOMAIN = {
    "domain": "logistica",
    "node_type": "unipersonal",  # "unipersonal" | "org"
    "module_id": "logistica_aprendiz",
    "action_dictionary_file": str(ROOT / "content" / "action_dictionary_state.json"),
    "soft_dictionary_file": str(ROOT / "content" / "soft_dictionary_state.json"),
}

Path(PARAMS["local_store_dir"]).mkdir(parents=True, exist_ok=True)
Path(PARAMS["bundle_dir"]).mkdir(parents=True, exist_ok=True)
Path(PARAMS["data_dir"]).mkdir(parents=True, exist_ok=True)
Path("configs").mkdir(parents=True, exist_ok=True)

print("OK — PARAMS listo.")

OK — PARAMS listo.


---

## Bloque 1 — El Portal de Ingesta (El “Cucurucho” Local)
**Entrada de realidad**
- Escanea un QR (aquí: campo de texto para NodeID/QR).
- O sube un archivo (PDF/log/factura) con FileUpload.
- El sistema “despierta” el manifold del nodo leyendo historial local.

In [ ]:
# === Utilidades de estado local ===
import json
import time
import re
import hashlib
from pathlib import Path
from typing import Dict, Any

def load_state_dict(path: str, fallback: Dict[str, Any]) -> Dict[str, Any]:
    p = Path(path)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(fallback, ensure_ascii=False, indent=2), encoding="utf-8")
    return fallback

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return "sha256:" + h.hexdigest()

def node_store_path(node_id: str) -> Path:
    safe = re.sub(r"[^a-zA-Z0-9_\-\.]+", "_", node_id).strip("_")
    p = Path(PARAMS["local_store_dir"]) / "nodes" / safe
    p.mkdir(parents=True, exist_ok=True)
    return p

def load_node_history(node_id: str) -> Dict[str, Any]:
    p = node_store_path(node_id) / "history.json"
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return {
        "node_id": node_id,
        "phase_history": [],
        "R_history": [],
        "taximeter_history": [],
        "last_seen": None
    }

def save_node_history(node_id: str, hist: Dict[str, Any]) -> None:
    hist["last_seen"] = time.strftime("%Y-%m-%dT%H:%M:%S%z")
    p = node_store_path(node_id) / "history.json"
    p.write_text(json.dumps(hist, ensure_ascii=False, indent=2), encoding="utf-8")

print("OK — estado local.")

OK — estado local.


In [ ]:
# === UI de Ingesta ===

node_id_in = W.Text(
    description="NodeID/QR:",
    placeholder="ej. cliente_001 o qr:ABC123",
    layout=W.Layout(width="520px")
)

upload = W.FileUpload(
    accept="", multiple=False, description="Subir archivo",
    layout=W.Layout(width="220px")
)

use_input_csv = W.Checkbox(value=True, description="Usar INPUT_RNN_READY.csv")
btn_ingest = W.Button(description="Ingestar", button_style="primary")
out_ingest = W.Output()

def _save_upload_to_buffer(node_id: str) -> Optional[Path]:
    if not upload.value:
        return None
    item = list(upload.value.values())[0]
    filename = item.get("metadata", {}).get("name", "uploaded.bin")
    data = item["content"]
    buf_dir = node_store_path(node_id) / "buffer"
    buf_dir.mkdir(parents=True, exist_ok=True)
    fpath = buf_dir / filename
    with open(fpath, "wb") as f:
        f.write(data)
    return fpath

def on_ingest(_):
    with out_ingest:
        clear_output()
        node_id = node_id_in.value.strip()
        if not node_id:
            print("⚠️ Escribe un NodeID/QR.")
            return

        hist = load_node_history(node_id)
        print(f"✅ Nodo cargado: {node_id}")
        if hist["phase_history"]:
            print(f"   Historial fases (últimas 5): {hist['phase_history'][-5:]}")
        else:
            print("   Nodo nuevo: sin historial previo.")

        uploaded_path = _save_upload_to_buffer(node_id)
        if uploaded_path:
            print(f"✅ Archivo recibido: {uploaded_path.name}")
            print(f"   Digest: {sha256_file(uploaded_path)}")
        else:
            print("ℹ️ No se subió archivo.")

        csv_path = Path(PARAMS["backbone_input_file"])
        if use_input_csv.value and csv_path.exists():
            df = pd.read_csv(csv_path)
            print(f"✅ INPUT_RNN_READY.csv listo: shape={df.shape}")
        else:
            print("ℹ️ INPUT_RNN_READY.csv no disponible o desactivado.")

        save_node_history(node_id, hist)
        print("✅ Manifold 'despierto'.")

btn_ingest.on_click(on_ingest)
display(W.HBox([node_id_in, upload, use_input_csv, btn_ingest]))
display(out_ingest)

Output()

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

# Define the path using PARAMS
rnn_input_file_path = Path(PARAMS["backbone_input_file"])

# Create a dummy DataFrame with some numerical data
dummy_data = {
    'feature_1': np.random.rand(10) * 100,
    'feature_2': np.random.rand(10) * 50 + 10,
    'categorical_feature': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'A', 'B', 'C']
}
dummy_df = pd.DataFrame(dummy_data)

# Ensure the directory exists
rnn_input_file_path.parent.mkdir(parents=True, exist_ok=True)

# Save the DataFrame to CSV
dummy_df.to_csv(rnn_input_file_path, index=False)

print(f"✅ Dummy INPUT_RNN_READY.csv creado en: {rnn_input_file_path}")
display(dummy_df.head())

✅ Dummy INPUT_RNN_READY.csv creado en: /content/content/INPUT_RNN_READY.csv


,feature_1,feature_2,categorical_feature
0,47.031860,54.631223,A
1,33.349212,16.124691,B
2,62.283797,27.303609,A
3,69.696663,23.241341,C
4,29.848936,19.838213,B


---

## Bloque 2 — Carga de la Doble Hélice (Backbone + Cabezas)
- Carga `universal_rules.json`.
- Carga backbone (real o stub).
- Descomprime heads zip.
- Lee `multiceph_config.json` (si existe) o usa un ejemplo compatible.
- Ejecuta inferencia de fase + heads condicionadas.

In [ ]:
# @title
def load_json_if_exists(path: str, default: Any) -> Any:
    p = Path(path)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return default

UNIVERSAL = load_json_if_exists(PARAMS["universal_rules_file"], default={
    "schema_version": "1.0",
    "phase_system": {"canonical_phases": ["stable", "tension", "drift", "rupture"]},
    "abstention_policy": {"conditions": {"min_probability_threshold": 0.65, "min_top2_gap": 0.10}},
})

PHASES = UNIVERSAL["phase_system"]["canonical_phases"]
print("OK — universal_rules. Fases:", PHASES)

OK — universal_rules. Fases: ['stable', 'tension', 'drift', 'rupture']


In [ ]:
# @title
class BackboneStub:
    def infer(self, X: np.ndarray) -> Dict[str, Any]:
        if X.size == 0:
            probs = np.array([0.25,0.25,0.25,0.25], dtype=float)
        else:
            m = float(np.mean(X[-5:])) if X.shape[0] >= 5 else float(np.mean(X))
            base = np.array([0.25,0.25,0.25,0.25], dtype=float)
            bias = np.tanh(m)
            base[0] += max(0.0, -bias) * 0.20
            base[1] += max(0.0,  bias) * 0.10
            base[2] += max(0.0,  bias) * 0.05
            base[3] += max(0.0,  bias) * 0.25
            probs = base / base.sum()
        pred = int(np.argmax(probs))
        entropy = float(-np.sum(probs * np.log(probs + 1e-12)))
        top2 = np.sort(probs)[-2:]
        gap = float(top2[-1] - top2[-2])
        return {
            "phase_idx": pred,
            "phase_name": PHASES[pred],
            "phase_probs": {PHASES[i]: float(probs[i]) for i in range(4)},
            "entropy": entropy,
            "top2_gap": gap
        }

def load_backbone(backbone_dir: str):
    # TODO: conecta aquí tu loader real (torch/onnx/etc.)
    return BackboneStub()

BACKBONE = load_backbone(PARAMS["backbone_dir"])
print("OK — backbone cargado (stub-safe).")

OK — backbone cargado (stub-safe).


In [ ]:
# @title
HEADS_WORKDIR = Path(PARAMS["local_store_dir"]) / "_heads_workdir"
HEADS_WORKDIR.mkdir(parents=True, exist_ok=True)

def extract_heads_zip(zip_path: str) -> Path:
    zp = Path(zip_path)
    if not zp.exists():
        print("⚠️ heads_zip no existe. Se continuará sin heads reales.")
        return HEADS_WORKDIR
    if HEADS_WORKDIR.exists():
        shutil.rmtree(HEADS_WORKDIR)
    HEADS_WORKDIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(HEADS_WORKDIR)
    return HEADS_WORKDIR

DEFAULT_MULTICEPH_CONFIG = {
    "bundle_name": "multiceph_example_v1",
    "phase_gating": {"mode": "hard", "threshold": 0.7},
    "heads": [
        {
            "name": "trigo_band_invariante",
            "type": "TrigoBandHead",
            "target_asset": "TRIGO",
            "input_spec": {"dataset": "TRIGO.csv", "price_col": "price", "window": 60},
            "params": {"band_width_factor": 0.001},
            "phase_conditioning": {"active_in_phases": [0,3]}
        },
        {
            "name": "cobre_rango_techo",
            "type": "CobreRangeCeilingHead",
            "target_asset": "COBRE",
            "input_spec": {"dataset": "COBRE.csv", "price_col": "price", "window": 0},
            "params": {"ceiling_factor": 1.05},
            "phase_conditioning": {"active_in_phases": [3]}
        },
        {
            "name": "petroleo_canal_colas",
            "type": "PetroleoChannelTailsHead",
            "target_asset": "PETROLEO",
            "input_spec": {"dataset": "PETROLEO.csv", "price_col": "price", "window": 0},
            "params": {"percentile_low": 0.05, "percentile_high": 0.95},
            "phase_conditioning": {"active_in_phases": [3]}
        }
    ]
}

cfg_path = Path("configs") / "multiceph_config.json"
if cfg_path.exists():
    MULTICEPH = json.loads(cfg_path.read_text(encoding="utf-8"))
else:
    MULTICEPH = DEFAULT_MULTICEPH_CONFIG

extract_heads_zip(PARAMS["heads_zip"])
print("OK — heads workdir:", HEADS_WORKDIR)
print("OK — heads en config:", len(MULTICEPH.get("heads", [])))

OK — heads workdir: /content/local_store/_heads_workdir
OK — heads en config: 3


In [ ]:
# @title
def phase_clarity_ok(phase_probs: Dict[str,float], threshold_p: float, min_gap: float):
    probs = np.array([phase_probs[p] for p in PHASES], dtype=float)
    pmax = float(np.max(probs))
    top2 = np.sort(probs)[-2:]
    gap = float(top2[-1] - top2[-2])
    ok = (pmax >= threshold_p) and (gap >= min_gap)
    return ok, pmax, gap

def run_heads(phase_idx: int, phase_probs: Dict[str,float]) -> Dict[str, Any]:
    gating = MULTICEPH.get("phase_gating", {"mode": "hard", "threshold": 0.7})
    thr = float(gating.get("threshold", 0.7))
    ok_g, pmax, gap = phase_clarity_ok(
        phase_probs,
        threshold_p=thr,
        min_gap=float(UNIVERSAL["abstention_policy"]["conditions"]["min_top2_gap"])
    )
    artifacts = {"phase_gating": {"ok": ok_g, "pmax": pmax, "gap": gap, "threshold": thr}, "heads": []}

    for h in MULTICEPH.get("heads", []):
        active = phase_idx in h.get("phase_conditioning", {}).get("active_in_phases", [])
        if not active:
            continue
        if gating.get("mode","hard") == "hard" and not ok_g:
            continue

        typ = h.get("type")
        name = h.get("name")
        params = h.get("params", {})

        artifact = {"name": name, "type": typ, "target_asset": h.get("target_asset"), "params": params, "result": {}}

        # TODO: conectar implementaciones reales por `type`
        if typ == "TrigoBandHead":
            artifact["result"] = {"band_width_factor": float(params.get("band_width_factor", 0.001)), "status": "ok_stub"}
        elif typ == "CobreRangeCeilingHead":
            artifact["result"] = {"ceiling_factor": float(params.get("ceiling_factor", 1.05)), "status": "ok_stub"}
        elif typ == "PetroleoChannelTailsHead":
            artifact["result"] = {"p_low": float(params.get("percentile_low", 0.05)), "p_high": float(params.get("percentile_high", 0.95)), "status": "ok_stub"}
        else:
            artifact["result"] = {"status": "unknown_head_type_stub"}

        artifacts["heads"].append(artifact)

    return artifacts

def load_backbone_input_csv(path: str) -> np.ndarray:
    p = Path(path)
    if not p.exists():
        return np.zeros((0,0), dtype=float)
    df = pd.read_csv(p)
    num = df.select_dtypes(include=[np.number])
    if num.shape[1] == 0:
        return np.zeros((0,0), dtype=float)
    return num.to_numpy(dtype=float)

def compute_R_stub(phase_name: str, heads_artifacts: Dict[str,Any]) -> float:
    base = {"stable": 0.80, "tension": 0.60, "drift": 0.50, "rupture": 0.30}.get(phase_name, 0.50)
    penalty = min(0.20, 0.05 * len(heads_artifacts.get("heads", [])))
    return float(max(0.0, min(1.0, base - penalty)))

def run_inference_episode() -> Dict[str, Any]:
    X = load_backbone_input_csv(PARAMS["backbone_input_file"])
    inf = BACKBONE.infer(X)
    phase_idx = int(inf["phase_idx"])
    phase_name = inf["phase_name"]
    phase_probs = inf["phase_probs"]

    abst_thr = float(UNIVERSAL["abstention_policy"]["conditions"]["min_probability_threshold"])
    abst_gap = float(UNIVERSAL["abstention_policy"]["conditions"]["min_top2_gap"])
    ok_u, pmax, gap = phase_clarity_ok(phase_probs, abst_thr, abst_gap)

    heads_artifacts = run_heads(phase_idx, phase_probs)
    R = compute_R_stub(phase_name, heads_artifacts)

    return {
        "phase": {"idx": phase_idx, "name": phase_name, "probs": phase_probs},
        "clarity": {"ok": ok_u, "pmax": pmax, "top2_gap": gap, "entropy": inf["entropy"]},
        "heads_artifacts": heads_artifacts,
        "R_score": R
    }

print("OK — motor de inferencia listo.")

OK — motor de inferencia listo.


---

## Bloque 3 — Tensor de Conciliación Local (TCL)
Proyecta una acción:
- basada en fase
- usando diccionario de acciones expandible
- con **Freno de R** si la acción estresa resiliencia

In [ ]:
# @title
# load_state_dict function is now in cell 62c45ecd

ACTION_STATE = load_state_dict(DOMAIN["action_dictionary_file"], fallback={
    "schema_version": "1.0",
    "domain": DOMAIN["domain"],
    "actions_by_phase": {
        "stable": [{"action_id":"act_archive_milestone","label":"Archivar hito y esperar respuesta","type":"operational"}],
        "tension": [{"action_id":"act_prioritize_deadlines","label":"Priorizar plazos críticos","type":"operational"}],
        "drift": [{"action_id":"act_redefine_scope","label":"Redefinir alcance/plan","type":"strategic"}],
        "rupture": [{"action_id":"act_activate_crisis_mode","label":"Activar modo crisis","type":"strategic"}]
    }
})

SOFT_STATE = load_state_dict(DOMAIN["soft_dictionary_file"], fallback={
    "schema_version": "1.0",
    "domain": DOMAIN["domain"],
    "custom_soft_vars": []
})

def suggest_action(phase_name: str, R: float) -> Dict[str, Any]:
    actions = ACTION_STATE.get("actions_by_phase", {}).get(phase_name, [])
    if not actions:
        return {"suggested": None, "reason": "no_actions_for_phase", "R_brake": None}

    a = actions[0]
    brake = None
    if R < 0.45 and a.get("type") in ("strategic","executive"):
        brake = {"warning": True, "message": "Freno de R: esta acción puede estresar resiliencia. ¿proceder?", "R": R}

    return {"suggested": a, "reason": "min_energy_projection_stub", "R_brake": brake}

print("OK — TCL listo.")

OK — TCL listo.


---

## Bloque 4 — Interfaz de Diálogo (Ajuste y Confirmación)
- Aceptar sugerencia
- Corregir y registrar excepción (Memoria del Pasante)
- Si corrige, el sistema **expande diccionario** para la fase actual

In [ ]:
# @title
btn_run = W.Button(description="Correr episodio", button_style="success")
out_episode = W.Output()

accept = W.ToggleButtons(
    options=[("Aceptar sugerencia","accept"), ("Corregir","correct")],
    value="accept",
    description="Decisión:"
)
corr_action_text = W.Text(
    description="Acción humana:",
    placeholder="Describe la acción si corriges (ej. 'Cierre preventivo de zona por ahorro energético')",
    layout=W.Layout(width="740px")
)
obs_text = W.Textarea(
    description="Observaciones:",
    placeholder="Ej. 'tenía días sin dormir'; 'interlocutor no confiable'; 'suelo tiembla'...",
    layout=W.Layout(width="740px", height="120px")
)
btn_commit = W.Button(description="Confirmar y Registrar", button_style="primary")
out_commit = W.Output()

EPISODE_STATE = {"last": None}

def run_episode_ui(_):
    with out_episode:
        clear_output()
        node_id = node_id_in.value.strip() or "node_default"
        res = run_inference_episode()
        tcl = suggest_action(res["phase"]["name"], res["R_score"])

        EPISODE_STATE["last"] = {"node_id": node_id, "inference": res, "tcl": tcl}

        display(Markdown(f"### Nodo: `{node_id}`"))
        display(Markdown(f"**Fase:** `{res['phase']['name']}` — claridad ok: `{res['clarity']['ok']}` (pmax={res['clarity']['pmax']:.2f}, gap={res['clarity']['gap']:.2f})"))
        display(Markdown(f"**R:** `{res['R_score']:.2f}`"))

        if res["heads_artifacts"]["heads"]:
            display(Markdown("**Artefactos de cabezas:**"))
            display(pd.DataFrame([
                {"name":h["name"],"type":h["type"],"target_asset":h["target_asset"],"result":json.dumps(h["result"])}
                for h in res["heads_artifacts"]["heads"]
            ]))
        else:
            display(Markdown("_Sin heads activas (o gating cerró)._"))

        if tcl["suggested"]:
            display(Markdown(f"### TCL — Sugerencia\n- **Acción:** `{tcl['suggested']['label']}` (`{tcl['suggested']['action_id']}`)"))
        else:
            display(Markdown("### TCL — Sugerencia\n- **Sin sugerencia** (diccionario vacío para esta fase)."))

        if tcl["R_brake"]:
            display(Markdown(f"⚠️ **{tcl['R_brake']['message']}** (R={tcl['R_brake']['R']:.2f})"))

        display(Markdown("### Diálogo"))
        display(W.VBox([accept, corr_action_text, obs_text, btn_commit, out_commit]))
        print("✅ Episodio preparado para confirmación. Revisa los detalles y presiona 'Confirmar y Registrar' para finalizar.")

btn_run.on_click(run_episode_ui)
display(btn_run)
display(out_episode)

Button(button_style='success', description='Correr episodio', style=ButtonStyle())

Output()

---

## Bloque 5 — Registro y Metabolismo (Cierre del Ciclo)
- Actualiza historial del nodo
- Registra `DecisionEvent` en `learning_log.jsonl`
- Taxímetro (stub) + forecast inmediato
- Recibo de destrucción: limpia buffers sensibles del nodo

In [ ]:
def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "node_id": node_id,
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {"observations": obs, "soft_tags": soft_tags},
            "taximeter_snapshot": tax,
            "receipt_destruction": receipt
        }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")

OK — registro/metabolismo listo.


## Hooks para tus artefactos reales

1) **Backbone real**: reemplaza `BackboneStub` en `load_backbone()`.
2) **Heads reales**: conecta en `run_heads()` el dispatch por `type`.
3) **INPUT_RNN_READY.csv**: generado por tu MMI/Embudo en `data_assets/`.
4) **Abstención**: umbrales desde `universal_rules.json`.
5) **Registro**: `learning_log.jsonl` es tu unidad mínima para ajuste bimestral.

Este v0.1 es una base estable para tu “caja con agujeros”.

# Task
Implement a new soft variable learning mechanism in the `on_commit` function. This mechanism should detect a potential new soft variable when the backbone's clarity is insufficient (`inf['clarity']['ok']` is False), the professional corrects the suggested action (`accept.value == 'correct'`), and the `obs_text` contains a keyword or phrase not already in `soft_tags` or `SOFT_STATE['custom_soft_vars']`. If detected, prompt the user to confirm and classify the new variable by including a tag like `[NEW_SOFT:keyword:dimension]` in the `obs_text`. Upon confirmation, add the new soft variable (keyword and dimension) to `SOFT_STATE['custom_soft_vars']` and save it to the `soft_dictionary_file`. Finally, register a 'learning_event' in the `event` dictionary, setting `abstention = true` and including `new_soft_variable` (with keyword and dimension), `action_taken`, and a `hard_metrics_snapshot`, ensuring this process does not alter the phase, modify the backbone, or reduce thresholds.

## Analyze Soft Variable Extraction

### Subtask:
Review the `on_commit` function to understand how existing soft tags are extracted from the `obs_text` and how `SOFT_STATE` is structured.


## Analyze Soft Variable Extraction

### Subtask:
Review the `on_commit` function to understand how existing soft tags are extracted from the `obs_text` and how `SOFT_STATE` is structured.

#### Instructions
1. Examine the `on_commit` function to identify the code block responsible for processing the `obs_text` and extracting `soft_tags`. Note how keywords are identified and added to the `soft_tags` list.
2. Locate where `SOFT_STATE` is loaded using `load_state_dict` and inspect its fallback structure as defined in the `load_state_dict` function call within the `on_commit` function's scope, paying close attention to the `custom_soft_vars` key.
3. Identify how the `soft_dictionary_file` is referenced and used in relation to `SOFT_STATE`.

---

### Step 1: Examine soft tag extraction in `on_commit`

In the `on_commit` function (cell `d36a0692`), locate the following section:

```python
        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))
```

This code block takes the `obs_text` (observations provided by the user) and converts it to lowercase. It then iterates through a predefined list of keywords (`"no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"`). If any of these keywords are found in the `obs_text`, they are added to the `soft_tags` list after replacing spaces with underscores. This demonstrates a simple keyword-based extraction mechanism for soft variables.

### Step 2: Locate `SOFT_STATE` loading and its fallback structure

In cell `30da3589`, the `SOFT_STATE` variable is initialized using the `load_state_dict` function:

```python
SOFT_STATE = load_state_dict(DOMAIN["soft_dictionary_file"], fallback={
    "schema_version": "1.0",
    "domain": DOMAIN["domain"],
    "custom_soft_vars": []
})
```

The `load_state_dict` function is designed to load a JSON file if it exists at the specified `soft_dictionary_file` path. If the file does not exist, it creates the file with the `fallback` dictionary and returns this default structure.

The `fallback` structure for `SOFT_STATE` explicitly defines a `"custom_soft_vars": []` key, which is an empty list. This indicates that the `SOFT_STATE` is intended to store custom soft variables, likely to be populated or expanded dynamically, although the current `on_commit` function does not modify this `custom_soft_vars` list directly. Instead, it extracts `soft_tags` based on a hardcoded list of keywords.

### Step 3: Identify how `soft_dictionary_file` is referenced and used

The `soft_dictionary_file` is defined within the `DOMAIN` dictionary in cell `910d304f`:

```python
DOMAIN = {
    "domain": "abogado",
    "node_type": "unipersonal",  # "unipersonal" | "org"
    "module_id": "abogado_aprendiz",
    "action_dictionary_file": str(ROOT / "configs" / "action_dictionary_state.json"),
    "soft_dictionary_file": str(ROOT / "configs" / "soft_dictionary_state.json"),
}
```

As seen in Step 2, this path (`DOMAIN["soft_dictionary_file"]`) is passed to the `load_state_dict` function when initializing `SOFT_STATE` in cell `30da3589`. The `load_state_dict` function uses this path to determine if a `soft_dictionary_state.json` file exists in the `configs/` directory. If it exists, the function loads the JSON content from this file into `SOFT_STATE`. If the file does not exist, it creates the file at this path with the specified `fallback` content, ensuring a persistent storage mechanism for the `SOFT_STATE`.

## Detect New Soft Variable Keywords

### Subtask:
Implement logic within the `on_commit` function to identify conditions for a new soft variable: 1) backbone had insufficient clarity (`inf['clarity']['ok']` is False), 2) professional took an unsuggested action (`accept.value == 'correct'`), and 3) the `obs_text` contains a keyword or phrase not present in the current `soft_tags` list or `SOFT_STATE['custom_soft_vars']`. This step will focus on finding the *potential* new soft variable from the observation text.


**Reasoning**:
The subtask requires modifying the `on_commit` function to detect new soft variable keywords under specific conditions. I will update the `on_commit` function with the requested logic for extracting, normalizing, and checking for new soft variable keywords.



In [ ]:
def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "node_id": node_id,
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags,
                "extracted_new_soft_keyword": extracted_new_soft_keyword, # Added for new detection
                "extracted_new_soft_dimension": extracted_new_soft_dimension # Added for new detection
            },
            "taximeter_snapshot": tax,
            "receipt_destruction": receipt
        }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")

NameError: name 'Dict' is not defined

**Reasoning**:
The previous code block failed with a `NameError` because `Dict` and `Any` from the `typing` module were not imported within the context of the function definitions. I need to add `from typing import Dict, Any` to the beginning of the code block to resolve this.



In [ ]:
from typing import Dict, Any

def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "node_id": node_id,
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags,
                "extracted_new_soft_keyword": extracted_new_soft_keyword, # Added for new detection
                "extracted_new_soft_dimension": extracted_new_soft_dimension # Added for new detection
            },
            "taximeter_snapshot": tax,
            "receipt_destruction": receipt
        }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")

NameError: name 'btn_commit' is not defined

**Reasoning**:
The previous code block failed with `NameError: name 'btn_commit' is not defined` because the `ipywidgets` objects `btn_commit`, `out_commit`, `accept`, `corr_action_text`, `obs_text`, and the `EPISODE_STATE` dictionary were not defined in the current execution scope, likely due to an incomplete notebook run or kernel restart. Additionally, the initial error `NameError: name 'Dict' is not defined` indicated that `Dict` and `Any` from the `typing` module were not imported. I will add the `from typing` import and include defensive initializations for the necessary widgets and `EPISODE_STATE` within this code block to ensure it runs correctly, while also preserving the logic for new soft variable detection that was already partially implemented.



In [ ]:
from typing import Dict, Any
import ipywidgets as W
from IPython.display import clear_output, Markdown, display # Ensure these are available if not from previous cells

# Defensive definitions for widgets and EPISODE_STATE
# In a typical notebook flow, these would be initialized in previous cells (e.g., fae81a51).
# These stubs ensure the code block can execute independently if the kernel state is reset.
try:
    # Attempt to access existing definitions to avoid re-creation if already present
    _ = btn_commit
    _ = out_commit
    _ = accept
    _ = corr_action_text
    _ = obs_text
    _ = EPISODE_STATE
except NameError:
    # Define minimal stubs if not found
    btn_commit = W.Button(description="Commit (Stub)")
    out_commit = W.Output()
    accept = W.ToggleButtons(options=[("Aceptar sugerencia","accept"), ("Corregir","correct")], value="accept")
    corr_action_text = W.Text(description="Acción humana:", placeholder="Describe la acción si corriges")
    obs_text = W.Textarea(description="Observaciones:", placeholder="Ej. 'tenía días sin dormir'")
    EPISODE_STATE = {"last": None}

# === Original functions and on_commit logic from cell d36a0692 ===

def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "node_id": node_id,
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags,
                "extracted_new_soft_keyword": extracted_new_soft_keyword, # Added for new detection
                "extracted_new_soft_dimension": extracted_new_soft_dimension # Added for new detection
            },
            "taximeter_snapshot": tax,
            "receipt_destruction": receipt
        }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")

OK — registro/metabolismo listo.


**Reasoning**:
The previous steps successfully implemented the detection of new soft variables. The next logical step, according to the subtask, is to actually add the new soft variable to `SOFT_STATE['custom_soft_vars']` and save this updated state to the `soft_dictionary_file`. Additionally, I need to modify the `event` dictionary to include a `learning_event` section when a new soft variable is identified.



In [ ]:
from typing import Dict, Any
import ipywidgets as W
from IPython.display import clear_output, Markdown, display

# Defensive definitions for widgets and EPISODE_STATE
try:
    _ = btn_commit
    _ = out_commit
    _ = accept
    _ = corr_action_text
    _ = obs_text
    _ = EPISODE_STATE
except NameError:
    btn_commit = W.Button(description="Commit (Stub)")
    out_commit = W.Output()
    accept = W.ToggleButtons(options=[("Aceptar sugerencia","accept"), ("Corregir","correct")], value="accept")
    corr_action_text = W.Text(description="Acción humana:", placeholder="Describe la acción si corriges")
    obs_text = W.Textarea(description="Observaciones:", placeholder="Ej. 'tenía días sin dormir'")
    EPISODE_STATE = {"last": None}

# === Original functions and on_commit logic from cell d36a0692 ===

def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        new_soft_variable_detected = None # Initialize a variable to hold the new soft variable info

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

                    # Add the new soft variable to SOFT_STATE and save
                    new_soft_var_entry = {"keyword": extracted_new_soft_keyword, "dimension": extracted_new_soft_dimension}
                    SOFT_STATE.setdefault("custom_soft_vars", []).append(new_soft_var_entry)
                    Path(DOMAIN["soft_dictionary_file"]).write_text(json.dumps(SOFT_STATE, ensure_ascii=False, indent=2), encoding="utf-8")
                    print(f"✅ Nueva variable soft '{extracted_new_soft_keyword}' guardada en el diccionario soft.")
                    new_soft_variable_detected = new_soft_var_entry # Store for learning event

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        # Prepare hard_metrics_snapshot for the learning event
        hard_metrics_snapshot = {
            "phase_name": inf["phase"]["name"],
            "clarity_ok": inf["clarity"]["ok"],
            "clarity_pmax": inf["clarity"]["pmax"],
            "clarity_gap": inf["clarity"]["gap"],
            "R_score": R
        }

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "node_id": node_id,
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags
            },
            "taximeter_snapshot": tax,
            "receipt_destruction": receipt
        }

        # Add learning_event if a new soft variable was detected
        if new_soft_variable_detected:
            event["learning_event"] = {
                "abstention": True,
                "new_soft_variable": new_soft_variable_detected,
                "action_taken": action_executed,
                "hard_metrics_snapshot": hard_metrics_snapshot
            }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Diccionario soft:", DOMAIN["soft_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")

OK — registro/metabolismo listo.


## Final Task

### Subtask:
Verify that the new soft variable learning process is correctly integrated, detects new terms, updates the dictionary, and logs the learning events as specified, without impacting the backbone or phase determination.


## Summary:

### Q&A
The new soft variable learning process is correctly integrated:
*   **Detection:** The process accurately detects new terms when `inf['clarity']['ok']` is `False`, the professional corrects the suggested action (`accept.value == 'correct'`), and the `obs_text` contains a `[NEW_SOFT:keyword:dimension]` tag not already present in `soft_tags` or `SOFT_STATE['custom_soft_vars']`.
*   **Dictionary Update:** Upon detection and confirmation (via the `[NEW_SOFT]` tag), the new soft variable (keyword and dimension) is successfully added to `SOFT_STATE['custom_soft_vars']` and persisted in the `soft_dictionary_file`.
*   **Learning Event Logging:** A `learning_event` is correctly registered in the event dictionary, containing `abstention = true`, `new_soft_variable` (with keyword and dimension), `action_taken`, and a `hard_metrics_snapshot`.
*   **Impact on Backbone/Phase:** The implementation strictly adheres to the requirement of not impacting the backbone or phase determination, as no code within the provided solution modifies these components or reduces thresholds.

### Data Analysis Key Findings

*   **Initial Soft Tag Extraction:** The `on_commit` function initially extracts soft tags from `obs_text` by checking for hardcoded keywords (e.g., "no dormir", "interlocutor") and adding them to a `soft_tags` list, normalizing keywords by replacing spaces with underscores.
*   **`SOFT_STATE` Structure and Persistence:** `SOFT_STATE` is loaded from `soft_dictionary_file` and initialized with a fallback structure including `"custom_soft_vars": []`, indicating a design for dynamically learned variables. The `soft_dictionary_file` (`configs/soft_dictionary_state.json`) is used for persistent storage of `SOFT_STATE`.
*   **Conditions for New Soft Variable Detection:** A new soft variable is considered for learning if three conditions are met:
    1.  The backbone's clarity is insufficient (`inf['clarity']['ok']` is `False`).
    2.  The professional corrects the suggested action (`accept.value` is `"correct"`).
    3.  The `obs_text` contains a user-provided tag in the format `[NEW_SOFT:keyword:dimension]`.
*   **Keyword Validation and Normalization:** When a `[NEW_SOFT]` tag is found, the extracted keyword is normalized (converted to lowercase and spaces replaced by underscores). It is then checked against existing `soft_tags` and `SOFT_STATE['custom_soft_vars']` to ensure uniqueness.
*   **Dynamic Dictionary Update:** If a unique new soft variable is detected, it is added as a dictionary `{"keyword": ..., "dimension": ...}` to the `SOFT_STATE['custom_soft_vars']` list, and `SOFT_STATE` is immediately saved to the `soft_dictionary_file`.
*   **Learning Event Logging:** Upon successful detection and storage of a new soft variable, a `learning_event` is appended to the main event log. This event includes:
    *   `"abstention": True`
    *   `"new_soft_variable"` containing the learned keyword and its dimension.
    *   `"action_taken"` detailing the action corrected by the user.
    *   `"hard_metrics_snapshot"` capturing the state of backbone metrics (phase name, clarity status, pmax, gap, R\_score) at the time of learning.
*   **Non-Impact on Core Logic:** The implemented mechanism specifically adds and logs new soft variables without modifying the core backbone inference, phase determination, or existing thresholds.

### Insights or Next Steps
*   **Refine User Input for New Variables:** Consider providing more structured input mechanisms for new soft variables beyond a free-text tag, such as dedicated input fields, to improve user experience and reduce parsing errors.
*   **Automated Keyword Detection Enhancement:** Explore natural language processing (NLP) techniques to automatically suggest potential new soft variable keywords and dimensions from `obs_text` when clarity is low and user correction occurs, rather than relying solely on explicit `[NEW_SOFT]` tags.


# Task
Create a stub function named `hard_metrics_snapshot_stub` that simulates capturing normalized values and key derivatives of 'hard variables' (e.g., `cash_norm`, `volatility_norm`, `utilization_norm`, `cash_d7`, `util_d7`). This stub will provide the `hard_metrics_snapshot` for the `decision_event` and `learning_event`.

## Define Hard Metrics Snapshot Stub

### Subtask:
Create a stub function named `hard_metrics_snapshot_stub` that simulates capturing normalized values and key derivatives of 'hard variables' as specified (e.g., `cash_norm`, `volatility_norm`, `utilization_norm`, `cash_d7`, `util_d7`). This stub will provide the `hard_metrics_snapshot` for the `decision_event` and `learning_event`.


**Reasoning**:
The subtask requires defining a stub function named `hard_metrics_snapshot_stub` that simulates capturing normalized values and key derivatives of 'hard variables'. I will create a new code cell for this function.



In [ ]:
def hard_metrics_snapshot_stub() -> Dict[str, Any]:
    """
    Simulates capturing normalized values and key derivatives of 'hard variables'.
    Returns:
        Dict[str, Any]: A dictionary with simulated hard metrics.
    """
    return {
        "cash_norm": float(np.random.rand()),          # Normalized cash (0-1)
        "volatility_norm": float(np.random.rand()),    # Normalized volatility (0-1)
        """utilization_norm""": float(np.random.rand()), # Normalized utilization (0-1)
        "cash_d7": float(np.random.uniform(-0.1, 0.1)), # 7-day derivative of cash
        "util_d7": float(np.random.uniform(-0.05, 0.05)) # 7-day derivative of utilization
    }

print("OK — hard_metrics_snapshot_stub defined.")

OK — hard_metrics_snapshot_stub defined.


**Reasoning**:
The `hard_metrics_snapshot_stub` function has been defined. Now, I need to integrate this function into the `on_commit` function to provide the `hard_metrics_snapshot` for both the `decision_event` (main event) and the `learning_event` as specified in the task.



In [ ]:
from typing import Dict, Any
import ipywidgets as W
from IPython.display import clear_output, Markdown, display

# Defensive definitions for widgets and EPISODE_STATE
try:
    _ = btn_commit
    _ = out_commit
    _ = accept
    _ = corr_action_text
    _ = obs_text
    _ = EPISODE_STATE
except NameError:
    btn_commit = W.Button(description="Commit (Stub)")
    out_commit = W.Output()
    accept = W.ToggleButtons(options=[("Aceptar sugerencia","accept"), ("Corregir","correct")], value="accept")
    corr_action_text = W.Text(description="Acción humana:", placeholder="Describe la acción si corriges")
    obs_text = W.Textarea(description="Observaciones:", placeholder="Ej. 'tenía días sin dormir'")
    EPISODE_STATE = {"last": None}

# === Original functions and on_commit logic from cell d36a0692 ===

def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def hard_metrics_snapshot_stub() -> Dict[str, Any]:
    """
    Simulates capturing normalized values and key derivatives of 'hard variables'.
    Returns:
        Dict[str, Any]: A dictionary with simulated hard metrics.
    """
    return {
        "cash_norm": float(np.random.rand()),          # Normalized cash (0-1)
        "volatility_norm": float(np.random.rand()),    # Normalized volatility (0-1)
        "utilization_norm": float(np.random.rand()), # Normalized utilization (0-1)
        "cash_d7": float(np.random.uniform(-0.1, 0.1)), # 7-day derivative of cash
        "util_d7": float(np.random.uniform(-0.05, 0.05)) # 7-day derivative of utilization
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        new_soft_variable_detected = None # Initialize a variable to hold the new soft variable info

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

                    # Add the new soft variable to SOFT_STATE and save
                    new_soft_var_entry = {"keyword": extracted_new_soft_keyword, "dimension": extracted_new_soft_dimension}
                    SOFT_STATE.setdefault("custom_soft_vars", []).append(new_soft_var_entry)
                    Path(DOMAIN["soft_dictionary_file"]).write_text(json.dumps(SOFT_STATE, ensure_ascii=False, indent=2), encoding="utf-8")
                    print(f"✅ Nueva variable soft '{extracted_new_soft_keyword}' guardada en el diccionario soft.")
                    new_soft_variable_detected = new_soft_var_entry # Store for learning event

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        # Get the hard metrics snapshot
        current_hard_metrics_snapshot = hard_metrics_snapshot_stub()

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "node_id": node_id,
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags,
                "extracted_new_soft_keyword": extracted_new_soft_keyword,
                "extracted_new_soft_dimension": extracted_new_soft_dimension
            },
            "taximeter_snapshot": tax,
            "hard_metrics_snapshot": current_hard_metrics_snapshot, # Added to the main event
            "receipt_destruction": receipt
        }

        # Add learning_event if a new soft variable was detected
        if new_soft_variable_detected:
            event["learning_event"] = {
                "abstention": True,
                "new_soft_variable": new_soft_variable_detected,
                "action_taken": action_executed,
                "hard_metrics_snapshot": current_hard_metrics_snapshot # Use the same snapshot
            }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Diccionario soft:", DOMAIN["soft_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")

OK — registro/metabolismo listo.


## Update Decision Event Structure

### Subtask:
Modify the `on_commit` function to update the main `event` dictionary to align with the `decision_event` specification. This includes: adding `module_id` from `DOMAIN`, explicitly defining `abstained` (based on `inf['clarity']['ok']`), `action_suggested_id` (if a suggestion was made), `action_new` (based on `action_known`), `hard_metrics_snapshot` using the new stub, `soft_context` including the structure of `custom_soft_vars`, and an `outcome_stub` field.


**Reasoning**:
The subtask requires modifying the `on_commit` function to align the main `event` dictionary with the `decision_event` specification. I will update the `event` dictionary with the new fields: `module_id`, `abstained`, `action_suggested_id`, `action_new`, `soft_context` (including `custom_soft_vars` and removing temporary fields), and `outcome_stub`.



In [ ]:
from typing import Dict, Any
import ipywidgets as W
from IPython.display import clear_output, Markdown, display

# Defensive definitions for widgets and EPISODE_STATE
try:
    _ = btn_commit
    _ = out_commit
    _ = accept
    _ = corr_action_text
    _ = obs_text
    _ = EPISODE_STATE
except NameError:
    btn_commit = W.Button(description="Commit (Stub)")
    out_commit = W.Output()
    accept = W.ToggleButtons(options=[("Aceptar sugerencia","accept"), ("Corregir","correct")], value="accept")
    corr_action_text = W.Text(description="Acción humana:", placeholder="Describe la acción si corriges")
    obs_text = W.Textarea(description="Observaciones:", placeholder="Ej. 'tenía días sin dormir'")
    EPISODE_STATE = {"last": None}

# === Original functions and on_commit logic ===

def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def hard_metrics_snapshot_stub() -> Dict[str, Any]:
    """
    Simulates capturing normalized values and key derivatives of 'hard variables'.
    Returns:
        Dict[str, Any]: A dictionary with simulated hard metrics.
    """
    return {
        "cash_norm": float(np.random.rand()),          # Normalized cash (0-1)
        "volatility_norm": float(np.random.rand()),    # Normalized volatility (0-1)
        "utilization_norm": float(np.random.rand()), # Normalized utilization (0-1)
        "cash_d7": float(np.random.uniform(-0.1, 0.1)), # 7-day derivative of cash
        "util_d7": float(np.random.uniform(-0.05, 0.05)) # 7-day derivative of utilization
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        new_soft_variable_detected = None # Initialize a variable to hold the new soft variable info

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

                    # Add the new soft variable to SOFT_STATE and save
                    new_soft_var_entry = {"keyword": extracted_new_soft_keyword, "dimension": extracted_new_soft_dimension}
                    SOFT_STATE.setdefault("custom_soft_vars", []).append(new_soft_var_entry)
                    Path(DOMAIN["soft_dictionary_file"]).write_text(json.dumps(SOFT_STATE, ensure_ascii=False, indent=2), encoding="utf-8")
                    print(f"✅ Nueva variable soft '{extracted_new_soft_keyword}' guardada en el diccionario soft.")
                    new_soft_variable_detected = new_soft_var_entry # Store for learning event

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        # Get the hard metrics snapshot
        current_hard_metrics_snapshot = hard_metrics_snapshot_stub()

        action_suggested_id = tcl["suggested"]["action_id"] if tcl["suggested"] else None

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "module_id": DOMAIN["module_id"], # Added module_id
            "node_id": node_id,
            "abstained": not inf['clarity']['ok'], # Added abstained
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "action_suggested_id": action_suggested_id, # Added action_suggested_id
                "action_new": not action_known, # Added action_new
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags,
                "custom_soft_vars": SOFT_STATE.get("custom_soft_vars", []) # Updated soft_context
            },
            "taximeter_snapshot": tax,
            "hard_metrics_snapshot": current_hard_metrics_snapshot, # Added hard_metrics_snapshot to main event
            "outcome_stub": "pending_feedback", # Added outcome_stub
            "receipt_destruction": receipt
        }

        # Remove temporary extracted_new_soft_keyword and extracted_new_soft_dimension if present
        if "extracted_new_soft_keyword" in event["soft_context"]:
            del event["soft_context"]["extracted_new_soft_keyword"]
        if "extracted_new_soft_dimension" in event["soft_context"]:
            del event["soft_context"]["extracted_new_soft_dimension"]

        # Add learning_event if a new soft variable was detected
        if new_soft_variable_detected:
            event["learning_event"] = {
                "abstention": True,
                "new_soft_variable": new_soft_variable_detected,
                "action_taken": action_executed,
                "hard_metrics_snapshot": current_hard_metrics_snapshot # Use the same snapshot
            }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Diccionario soft:", DOMAIN["soft_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")


OK — registro/metabolismo listo.


## Implement Abstention Event Details

### Subtask:
Within the `on_commit` function, if `abstained` is true (i.e., `not inf['clarity']['ok']`), add an `abstention_details` sub-dictionary to the main `event`. This sub-dictionary will include `phase_entropy` (`inf['clarity']['entropy']`), `p_phase_top1` and `p_phase_top2` (derived from `inf['clarity']['probs']` and `inf['clarity']['gap']`), and `reason_codes` (derived from clarity metrics, e.g., 'low_separation', 'high_transition_entropy').


**Reasoning**:
The subtask requires modifying the `on_commit` function to add abstention details to the event dictionary when abstention occurs. I will add the logic to calculate `phase_probs_values`, `p_phase_top1`, `p_phase_top2`, and `reason_codes` and then include them in an `abstention_details` sub-dictionary within the `event`.



In [ ]:
from typing import Dict, Any
import ipywidgets as W
from IPython.display import clear_output, Markdown, display
import numpy as np # Import numpy for calculations of entropy and log

# Defensive definitions for widgets and EPISODE_STATE
try:
    _ = btn_commit
    _ = out_commit
    _ = accept
    _ = corr_action_text
    _ = obs_text
    _ = EPISODE_STATE
except NameError:
    btn_commit = W.Button(description="Commit (Stub)")
    out_commit = W.Output()
    accept = W.ToggleButtons(options=[("Aceptar sugerencia","accept"), ("Corregir","correct")], value="accept")
    corr_action_text = W.Text(description="Acción humana:", placeholder="Describe la acción si corriges")
    obs_text = W.Textarea(description="Observaciones:", placeholder="Ej. 'tenía días sin dormir'")
    EPISODE_STATE = {"last": None}

# Assuming UNIVERSAL and PHASES are globally defined from previous cells
# If not, add defensive stubs here for local execution
try:
    _ = UNIVERSAL
    _ = PHASES
except NameError:
    UNIVERSAL = {
        "schema_version": "1.0",
        "phase_system": {"canonical_phases": ["stable", "tension", "drift", "rupture"]},
        "abstention_policy": {"conditions": {"min_probability_threshold": 0.65, "min_top2_gap": 0.10}},
    }
    PHASES = UNIVERSAL["phase_system"]["canonical_phases"]

# === Original functions and on_commit logic ===

def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def hard_metrics_snapshot_stub() -> Dict[str, Any]:
    """
    Simulates capturing normalized values and key derivatives of 'hard variables'.
    Returns:
        Dict[str, Any]: A dictionary with simulated hard metrics.
    """
    return {
        "cash_norm": float(np.random.rand()),          # Normalized cash (0-1)
        "volatility_norm": float(np.random.rand()),    # Normalized volatility (0-1)
        "utilization_norm": float(np.random.rand()), # Normalized utilization (0-1)
        "cash_d7": float(np.random.uniform(-0.1, 0.1)), # 7-day derivative of cash
        "util_d7": float(np.random.uniform(-0.05, 0.05)) # 7-day derivative of utilization
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        new_soft_variable_detected = None # Initialize a variable to hold the new soft variable info

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

                    # Add the new soft variable to SOFT_STATE and save
                    new_soft_var_entry = {"keyword": extracted_new_soft_keyword, "dimension": extracted_new_soft_dimension}
                    SOFT_STATE.setdefault("custom_soft_vars", []).append(new_soft_var_entry)
                    Path(DOMAIN["soft_dictionary_file"]).write_text(json.dumps(SOFT_STATE, ensure_ascii=False, indent=2), encoding="utf-8")
                    print(f"✅ Nueva variable soft '{extracted_new_soft_keyword}' guardada en el diccionario soft.")
                    new_soft_variable_detected = new_soft_var_entry # Store for learning event

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        # Get the hard metrics snapshot
        current_hard_metrics_snapshot = hard_metrics_snapshot_stub()

        action_suggested_id = tcl["suggested"]["action_id"] if tcl["suggested"] else None

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "module_id": DOMAIN["module_id"], # Added module_id
            "node_id": node_id,
            "abstained": not inf['clarity']['ok'], # Added abstained
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "action_suggested_id": action_suggested_id, # Added action_suggested_id
                "action_new": not action_known, # Added action_new
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags,
                "custom_soft_vars": SOFT_STATE.get("custom_soft_vars", []) # Updated soft_context
            },
            "taximeter_snapshot": tax,
            "hard_metrics_snapshot": current_hard_metrics_snapshot, # Added hard_metrics_snapshot to main event
            "outcome_stub": "pending_feedback", # Added outcome_stub
            "receipt_destruction": receipt
        }

        # Remove temporary extracted_new_soft_keyword and extracted_new_soft_dimension if present
        if "extracted_new_soft_keyword" in event["soft_context"]:
            del event["soft_context"]["extracted_new_soft_keyword"]
        if "extracted_new_soft_dimension" in event["soft_context"]:
            del event["soft_context"]["extracted_new_soft_dimension"]

        # Add abstention_details if abstained is true
        if event['abstained']:
            phase_probs_values = list(inf['phase']['probs'].values())
            p_phase_top1 = float(np.max(phase_probs_values))
            # Ensure there are at least two probabilities for top2, otherwise set to None or 0
            p_phase_top2 = float(np.sort(phase_probs_values)[-2]) if len(phase_probs_values) >= 2 else 0.0

            reason_codes = []
            min_top2_gap_threshold = UNIVERSAL['abstention_policy']['conditions']['min_top2_gap']
            min_prob_threshold = UNIVERSAL['abstention_policy']['conditions']['min_probability_threshold']
            max_possible_entropy = np.log(len(PHASES)) if len(PHASES) > 0 else 0.0 # Entropy for uniform distribution

            if inf['clarity']['gap'] < min_top2_gap_threshold:
                reason_codes.append('low_separation')
            if inf['clarity']['pmax'] < min_prob_threshold:
                reason_codes.append('low_confidence')
            # Example threshold for high entropy, can be tuned.
            # If entropy is close to maximum possible entropy (uniform distribution), it implies high uncertainty.
            if inf['clarity']['entropy'] > 0.7 * max_possible_entropy and max_possible_entropy > 0:
                reason_codes.append('high_transition_entropy')

            event['abstention_details'] = {
                'phase_entropy': inf['clarity']['entropy'],
                'p_phase_top1': p_phase_top1,
                'p_phase_top2': p_phase_top2,
                'reason_codes': reason_codes
            }

        # Add learning_event if a new soft variable was detected
        if new_soft_variable_detected:
            event["learning_event"] = {
                "abstention": True,
                "new_soft_variable": new_soft_variable_detected,
                "action_taken": action_executed,
                "hard_metrics_snapshot": current_hard_metrics_snapshot # Use the same snapshot
            }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Diccionario soft:", DOMAIN["soft_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")

OK — registro/metabolismo listo.


## Implement New Action Event Details

### Subtask:
Within the `on_commit` function, if `action_new` is true (i.e., `not action_known`), add a `new_action_details` sub-dictionary to the main `event`. This sub-dictionary will include `new_action_id`, `label`, `phase_binding` (the phase in which it was created), `type`, a placeholder `impact_vector_est` (stub), and `created_from_abst` (indicating if it was created due to an abstention event).


## Implement New Action Event Details

### Subtask:
Within the `on_commit` function, if `action_new` is true (i.e., `not action_known`), add a `new_action_details` sub-dictionary to the main `event`. This sub-dictionary will include `new_action_id`, `label`, `phase_binding` (the phase in which it was created), `type`, a placeholder `impact_vector_est` (stub), and `created_from_abst` (indicating if it was created due to an abstention event).

#### Instructions
1. Locate the `on_commit` function.
2. After the main `event` dictionary is constructed, add a conditional block to check if `action_new` is true (i.e., `not action_known`).
3. Inside this conditional block, create a new dictionary called `new_action_details`.
4. Populate `new_action_details` with the following keys and values:
    *   `"new_action_id"`: Use `action_executed["action_id"]`.
    *   `"label"`: Use `action_executed["label"]`.
    *   `"phase_binding"`: Use the `phase` variable.
    *   `"type"`: Use `action_executed["type"]`.
    *   `"impact_vector_est"`: Set this to a placeholder stub, for example, `[0.0, 0.0, 0.0, 0.0]`.
    *   `"created_from_abst"`: Set this to `event['abstained']`.
5. Add the `new_action_details` dictionary to the main `event` dictionary under the key `"new_action_details"`.

**Reasoning**:
The subtask requires modifying the `on_commit` function to add `new_action_details` when `action_new` is true. I will update the `on_commit` function with the new conditional block and dictionary.



In [ ]:
from typing import Dict, Any
import ipywidgets as W
from IPython.display import clear_output, Markdown, display
import numpy as np # Import numpy for calculations of entropy and log

# Defensive definitions for widgets and EPISODE_STATE
try:
    _ = btn_commit
    _ = out_commit
    _ = accept
    _ = corr_action_text
    _ = obs_text
    _ = EPISODE_STATE
except NameError:
    btn_commit = W.Button(description="Commit (Stub)")
    out_commit = W.Output()
    accept = W.ToggleButtons(options=[("Aceptar sugerencia","accept"), ("Corregir","correct")], value="accept")
    corr_action_text = W.Text(description="Acción humana:", placeholder="Describe la acción si corriges")
    obs_text = W.Textarea(description="Observaciones:", placeholder="Ej. 'tenía días sin dormir'")
    EPISODE_STATE = {"last": None}

# Assuming UNIVERSAL and PHASES are globally defined from previous cells
# If not, add defensive stubs here for local execution
try:
    _ = UNIVERSAL
    _ = PHASES
except NameError:
    UNIVERSAL = {
        "schema_version": "1.0",
        "phase_system": {"canonical_phases": ["stable", "tension", "drift", "rupture"]},
        "abstention_policy": {"conditions": {"min_probability_threshold": 0.65, "min_top2_gap": 0.10}},
    }
    PHASES = UNIVERSAL["phase_system"]["canonical_phases"]

# === Original functions and on_commit logic ===

def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def hard_metrics_snapshot_stub() -> Dict[str, Any]:
    """
    Simulates capturing normalized values and key derivatives of 'hard variables'.
    Returns:
        Dict[str, Any]: A dictionary with simulated hard metrics.
    """
    return {
        "cash_norm": float(np.random.rand()),          # Normalized cash (0-1)
        "volatility_norm": float(np.random.rand()),    # Normalized volatility (0-1)
        "utilization_norm": float(np.random.rand()), # Normalized utilization (0-1)
        "cash_d7": float(np.random.uniform(-0.1, 0.1)), # 7-day derivative of cash
        "util_d7": float(np.random.uniform(-0.05, 0.05)) # 7-day derivative of utilization
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        new_soft_variable_detected = None # Initialize a variable to hold the new soft variable info

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

                    # Add the new soft variable to SOFT_STATE and save
                    new_soft_var_entry = {"keyword": extracted_new_soft_keyword, "dimension": extracted_new_soft_dimension}
                    SOFT_STATE.setdefault("custom_soft_vars", []).append(new_soft_var_entry)
                    Path(DOMAIN["soft_dictionary_file"]).write_text(json.dumps(SOFT_STATE, ensure_ascii=False, indent=2), encoding="utf-8")
                    print(f"✅ Nueva variable soft '{extracted_new_soft_keyword}' guardada en el diccionario soft.")
                    new_soft_variable_detected = new_soft_var_entry # Store for learning event

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        # Get the hard metrics snapshot
        current_hard_metrics_snapshot = hard_metrics_snapshot_stub()

        action_suggested_id = tcl["suggested"]["action_id"] if tcl["suggested"] else None

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "module_id": DOMAIN["module_id"], # Added module_id
            "node_id": node_id,
            "abstained": not inf['clarity']['ok'], # Added abstained
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "action_suggested_id": action_suggested_id, # Added action_suggested_id
                "action_new": not action_known, # Added action_new
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags,
                "custom_soft_vars": SOFT_STATE.get("custom_soft_vars", []) # Updated soft_context
            },
            "taximeter_snapshot": tax,
            "hard_metrics_snapshot": current_hard_metrics_snapshot, # Added hard_metrics_snapshot to main event
            "outcome_stub": "pending_feedback", # Added outcome_stub
            "receipt_destruction": receipt
        }

        # Remove temporary extracted_new_soft_keyword and extracted_new_soft_dimension if present
        if "extracted_new_soft_keyword" in event["soft_context"]:
            del event["soft_context"]["extracted_new_soft_keyword"]
        if "extracted_new_soft_dimension" in event["soft_context"]:
            del event["soft_context"]["extracted_new_soft_dimension"]

        # Add abstention_details if abstained is true
        if event['abstained']:
            phase_probs_values = list(inf['phase']['probs'].values())
            p_phase_top1 = float(np.max(phase_probs_values))
            # Ensure there are at least two probabilities for top2, otherwise set to None or 0
            p_phase_top2 = float(np.sort(phase_probs_values)[-2]) if len(phase_probs_values) >= 2 else 0.0

            reason_codes = []
            min_top2_gap_threshold = UNIVERSAL['abstention_policy']['conditions']['min_top2_gap']
            min_prob_threshold = UNIVERSAL['abstention_policy']['conditions']['min_probability_threshold']
            max_possible_entropy = np.log(len(PHASES)) if len(PHASES) > 0 else 0.0 # Entropy for uniform distribution

            if inf['clarity']['gap'] < min_top2_gap_threshold:
                reason_codes.append('low_separation')
            if inf['clarity']['pmax'] < min_prob_threshold:
                reason_codes.append('low_confidence')
            # Example threshold for high entropy, can be tuned.
            # If entropy is close to maximum possible entropy (uniform distribution), it implies high uncertainty.
            if inf['clarity']['entropy'] > 0.7 * max_possible_entropy and max_possible_entropy > 0:
                reason_codes.append('high_transition_entropy')

            event['abstention_details'] = {
                'phase_entropy': inf['clarity']['entropy'],
                'p_phase_top1': p_phase_top1,
                'p_phase_top2': p_phase_top2,
                'reason_codes': reason_codes
            }

        # Add new_action_details if action_new is true
        if event["action_execution"]["action_new"]:
            event["new_action_details"] = {
                "new_action_id": action_executed["action_id"],
                "label": action_executed["label"],
                "phase_binding": phase,
                "type": action_executed["type"],
                "impact_vector_est": [0.0, 0.0, 0.0, 0.0],
                "created_from_abst": event['abstained']
            }

        # Add learning_event if a new soft variable was detected
        if new_soft_variable_detected:
            event["learning_event"] = {
                "abstention": True,
                "new_soft_variable": new_soft_variable_detected,
                "action_taken": action_executed,
                "hard_metrics_snapshot": current_hard_metrics_snapshot # Use the same snapshot
            }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "gap": inf["clarity"]["gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Diccionario soft:", DOMAIN["soft_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")


OK — registro/metabolismo listo.


## Integrate Learning Event with Abstention Context

### Subtask:
Review and ensure the `learning_event` (from the previous task for new soft variables) is correctly integrated. Specifically, ensure that its `abstention` field accurately reflects the model's clarity state, and ensure all required sub-fields (like `new_soft_variable`, `action_taken`, `hard_metrics_snapshot`) are consistently populated, respecting the non-alteration constraints for backbone, phase, and thresholds.


**Reasoning**:
The subtask requires modifying the `learning_event` within the `on_commit` function to accurately reflect the `abstention` status using `event['abstained']` and verifying the consistent population of other fields. I will update the relevant line in the `on_commit` function.



In [ ]:
from typing import Dict, Any
import ipywidgets as W
from IPython.display import clear_output, Markdown, display
import numpy as np # Import numpy for calculations of entropy and log

# Defensive definitions for widgets and EPISODE_STATE
try:
    _ = btn_commit
    _ = out_commit
    _ = accept
    _ = corr_action_text
    _ = obs_text
    _ = EPISODE_STATE
except NameError:
    btn_commit = W.Button(description="Commit (Stub)")
    out_commit = W.Output()
    accept = W.ToggleButtons(options=[("Aceptar sugerencia","accept"), ("Corregir","correct")], value="accept")
    corr_action_text = W.Text(description="Acción humana:", placeholder="Describe la acción si corriges")
    obs_text = W.Textarea(description="Observaciones:", placeholder="Ej. 'tenía días sin dormir'")
    EPISODE_STATE = {"last": None}

# Assuming UNIVERSAL and PHASES are globally defined from previous cells
# If not, add defensive stubs here for local execution
try:
    _ = UNIVERSAL
    _ = PHASES
except NameError:
    UNIVERSAL = {
        "schema_version": "1.0",
        "phase_system": {"canonical_phases": ["stable", "tension", "drift", "rupture"]},
        "abstention_policy": {"conditions": {"min_probability_threshold": 0.65, "min_top2_gap": 0.10}},
    }
    PHASES = UNIVERSAL["phase_system"]["canonical_phases"]

# === Original functions and on_commit logic ===

def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def next_event_id() -> str:
    return "evt_" + time.strftime("%Y%m%d_%H%M%S")

def destroy_sensitive_buffers(node_id: str) -> Dict[str, Any]:
    buf_dir = node_store_path(node_id) / "buffer"
    destroyed = []
    if buf_dir.exists():
        for fp in buf_dir.glob("*"):
            try:
                fp.unlink()
                destroyed.append(fp.name)
            except Exception:
                pass
        try:
            buf_dir.rmdir()
        except Exception:
            pass
    return {"destroyed_files": destroyed, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

def taximeter_stub(phase_name: str, R: float) -> Dict[str, Any]:
    base = {"stable": 0.25, "tension": 0.55, "drift": 0.65, "rupture": 0.85}.get(phase_name, 0.50)
    score = float(min(1.0, max(0.0, base + (0.45 - R))))
    delta_next = float(min(1.0, max(-1.0, (score - base))))
    return {
        "taximeter_id": "taximeter_core_local_cost",
        "cost_space": {"name": "local_operational_cost", "unit": "cost_index", "range": [0.0, 1.0]},
        "now": {"score": score, "delta_next": delta_next, "horizon_days_next": 7},
        "forecast": [
            {"target":"score","horizon_days":7,"p50": min(1.0, score+0.05), "p90": min(1.0, score+0.12)},
            {"target":"score","horizon_days":30,"p50": min(1.0, score+0.10), "p90": min(1.0, score+0.25)}
        ],
        "drivers_topk": [
            {"id":"geo_phase", "weight":0.40},
            {"id":"geo_R", "weight":0.35},
            {"id":"heads_active", "weight":0.25}
        ]
    }

def hard_metrics_snapshot_stub() -> Dict[str, Any]:
    """
    Simulates capturing normalized values and key derivatives of 'hard variables'.
    Returns:
        Dict[str, Any]: A dictionary with simulated hard metrics.
    """
    return {
        "cash_norm": float(np.random.rand()),          # Normalized cash (0-1)
        "volatility_norm": float(np.random.rand()),    # Normalized volatility (0-1)
        "utilization_norm": float(np.random.rand()), # Normalized utilization (0-1)
        "cash_d7": float(np.random.uniform(-0.1, 0.1)), # 7-day derivative of cash
        "util_d7": float(np.random.uniform(-0.05, 0.05)) # 7-day derivative of utilization
    }

def on_commit(_):
    with out_commit:
        clear_output()
        st = EPISODE_STATE.get("last")
        if not st:
            print("⚠️ Primero corre el episodio.")
            return

        node_id = st["node_id"]
        inf = st["inference"]
        tcl = st["tcl"]

        phase = inf["phase"]["name"]
        R = float(inf["R_score"])

        if accept.value == "accept" and tcl["suggested"]:
            action_executed = tcl["suggested"]
            user_accepted = True
            action_known = True
        else:
            label = corr_action_text.value.strip()
            if not label:
                print("⚠️ Si corriges, escribe la acción humana.")
                return
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()
            action_id = f"act_custom_{slug[:48]}"
            action_executed = {"action_id": action_id, "label": label, "type": "operational"}
            user_accepted = False
            action_known = False

            # Expand diccionario en fase actual
            ACTION_STATE.setdefault("actions_by_phase", {}).setdefault(phase, [])
            ACTION_STATE["actions_by_phase"][phase].insert(0, action_executed)
            Path(DOMAIN["action_dictionary_file"]).write_text(json.dumps(ACTION_STATE, ensure_ascii=False, indent=2), encoding="utf-8")

        obs = obs_text.value.strip()
        soft_tags = []
        if obs:
            for kw in ["no dormir","sin dormir","interlocutor","tiembla","alcohol","urgente","riesgo"]:
                if kw in obs.lower():
                    soft_tags.append(kw.replace(" ", "_"))

        new_soft_variable_detected = None # Initialize a variable to hold the new soft variable info

        extracted_new_soft_keyword = None
        extracted_new_soft_dimension = None

        # Conditions for a new soft variable: 1) insufficient clarity, 2) unsuggested action
        if not inf['clarity']['ok'] and not user_accepted:
            # Search for pattern like [NEW_SOFT:some_keyword:some_dimension]
            match = re.search(r'\[NEW_SOFT:([a-zA-Z0-9_]+):([a-zA-Z0-9_]+)\]', obs)
            if match:
                potential_keyword = match.group(1)
                potential_dimension = match.group(2)

                # Normalize the keyword
                normalized_keyword = potential_keyword.lower().replace(' ', '_')

                # Check if this keyword already exists in soft_tags
                keyword_in_soft_tags = normalized_keyword in soft_tags

                # Check if this keyword already exists in SOFT_STATE['custom_soft_vars']
                keyword_in_custom_vars = any(
                    item.get('keyword') == normalized_keyword
                    for item in SOFT_STATE.get('custom_soft_vars', [])
                )

                if not keyword_in_soft_tags and not keyword_in_custom_vars:
                    extracted_new_soft_keyword = normalized_keyword
                    extracted_new_soft_dimension = potential_dimension
                    print(f"✅ Potencial nueva variable soft detectada: {extracted_new_soft_keyword} ({extracted_new_soft_dimension})")

                    # Add the new soft variable to SOFT_STATE and save
                    new_soft_var_entry = {"keyword": extracted_new_soft_keyword, "dimension": extracted_new_soft_dimension}
                    SOFT_STATE.setdefault("custom_soft_vars", []).append(new_soft_var_entry)
                    Path(DOMAIN["soft_dictionary_file"]).write_text(json.dumps(SOFT_STATE, ensure_ascii=False, indent=2), encoding="utf-8")
                    print(f"✅ Nueva variable soft '{extracted_new_soft_keyword}' guardada en el diccionario soft.")
                    new_soft_variable_detected = new_soft_var_entry # Store for learning event

        tax = taximeter_stub(phase, R)
        receipt = destroy_sensitive_buffers(node_id)

        # Get the hard metrics snapshot
        current_hard_metrics_snapshot = hard_metrics_snapshot_stub()

        action_suggested_id = tcl["suggested"]["action_id"] if tcl["suggested"] else None

        event = {
            "event_id": next_event_id(),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
            "domain": DOMAIN["domain"],
            "module_id": DOMAIN["module_id"], # Added module_id
            "node_id": node_id,
            "abstained": not inf['clarity']['ok'], # Added abstained
            "backbone_inference": {"phase": phase, "phase_probs": inf["phase"]["probs"], "clarity": inf["clarity"], "R_score": R},
            "heads_artifacts": inf["heads_artifacts"],
            "tcl": tcl,
            "action_execution": {
                "action_known": action_known,
                "action_id": action_executed["action_id"],
                "action_label": action_executed["label"],
                "action_suggested_id": action_suggested_id, # Added action_suggested_id
                "action_new": not action_known, # Added action_new
                "user_accepted_suggestion": user_accepted,
                "completion_state": "completed",
                "instance_ref": None
            },
            "soft_context": {
                "observations": obs,
                "soft_tags": soft_tags,
                "custom_soft_vars": SOFT_STATE.get("custom_soft_vars", []) # Updated soft_context
            },
            "taximeter_snapshot": tax,
            "hard_metrics_snapshot": current_hard_metrics_snapshot, # Added hard_metrics_snapshot to main event
            "outcome_stub": "pending_feedback", # Added outcome_stub
            "receipt_destruction": receipt
        }

        # Remove temporary extracted_new_soft_keyword and extracted_new_soft_dimension if present
        if "extracted_new_soft_keyword" in event["soft_context"]:
            del event["soft_context"]["extracted_new_soft_keyword"]
        if "extracted_new_soft_dimension" in event["soft_context"]:
            del event["soft_context"]["extracted_new_soft_dimension"]

        # Add abstention_details if abstained is true
        if event['abstained']:
            phase_probs_values = list(inf['phase']['probs'].values())
            p_phase_top1 = float(np.max(phase_probs_values))
            # Ensure there are at least two probabilities for top2, otherwise set to None or 0
            p_phase_top2 = float(np.sort(phase_probs_values)[-2]) if len(phase_probs_values) >= 2 else 0.0

            reason_codes = []
            min_top2_gap_threshold = UNIVERSAL['abstention_policy']['conditions']['min_top2_gap']
            min_prob_threshold = UNIVERSAL['abstention_policy']['conditions']['min_probability_threshold']
            max_possible_entropy = np.log(len(PHASES)) if len(PHASES) > 0 else 0.0 # Entropy for uniform distribution

            if inf['clarity']['top2_gap'] < min_top2_gap_threshold:
                reason_codes.append('low_separation')
            if inf['clarity']['pmax'] < min_prob_threshold:
                reason_codes.append('low_confidence')
            # Example threshold for high entropy, can be tuned.
            # If entropy is close to maximum possible entropy (uniform distribution), it implies high uncertainty.
            if inf['clarity']['entropy'] > 0.7 * max_possible_entropy and max_possible_entropy > 0:
                reason_codes.append('high_transition_entropy')

            event['abstention_details'] = {
                'phase_entropy': inf['clarity']['entropy'],
                'p_phase_top1': p_phase_top1,
                'p_phase_top2': p_phase_top2,
                'reason_codes': reason_codes
            }

        # Add new_action_details if action_new is true
        if event["action_execution"]["action_new"]:
            event["new_action_details"] = {
                "new_action_id": action_executed["action_id"],
                "label": action_executed["label"],
                "phase_binding": phase,
                "type": action_executed["type"],
                "impact_vector_est": [0.0, 0.0, 0.0, 0.0],
                "created_from_abst": event['abstained']
            }

        # Add learning_event if a new soft variable was detected
        if new_soft_variable_detected:
            event["learning_event"] = {
                "abstention": event['abstained'], # Updated to reflect model's clarity state
                "new_soft_variable": new_soft_variable_detected,
                "action_taken": action_executed,
                "hard_metrics_snapshot": current_hard_metrics_snapshot # Use the same snapshot
            }

        append_jsonl(PARAMS["learning_log_file"], event)

        # Update node history
        hist = load_node_history(node_id)
        hist["phase_history"].append({"t": event["timestamp"], "phase": phase, "pmax": inf["clarity"]["pmax"], "top2_gap": inf["clarity"]["top2_gap"]})
        hist["R_history"].append({"t": event["timestamp"], "R": R})
        hist["taximeter_history"].append({"t": event["timestamp"], "taximeter_id": tax["taximeter_id"], "score": tax["now"]["score"]})
        save_node_history(node_id, hist)

        print("✅ Evento registrado:", event["event_id"])
        print("✅ Learning log:", PARAMS["learning_log_file"])
        print("✅ Diccionario acciones:", DOMAIN["action_dictionary_file"])
        print("✅ Diccionario soft:", DOMAIN["soft_dictionary_file"])
        print("✅ Recibo destrucción:", receipt)

btn_commit.on_click(on_commit)
print("OK — registro/metabolismo listo.")

OK — registro/metabolismo listo.


## Final Task

### Subtask:
Verify that the `on_commit` function now correctly logs `decision_event` with all specified fields, including conditional `abstention_details` and `new_action_details`, and ensures the `learning_event` structure is consistent within the `daily_learning_log`.


## Summary:

### Data Analysis Key Findings

*   The `hard_metrics_snapshot_stub` function was successfully created, simulating normalized hard variables (e.g., `cash_norm`, `volatility_norm`, `utilization_norm` between 0 and 1) and their 7-day derivatives (e.g., `cash_d7` from -0.1 to 0.1, `util_d7` from -0.05 to 0.05).
*   The `on_commit` function was updated to correctly populate the `decision_event` (main event log). This included adding `module_id`, `abstained` (based on inference clarity), `action_suggested_id`, `action_new`, and the newly generated `hard_metrics_snapshot`. The `soft_context` was also enhanced with `custom_soft_vars`, and an `outcome_stub` was introduced.
*   Conditional `abstention_details` are now included in the `decision_event` when `abstained` is true. This sub-dictionary contains `phase_entropy`, `p_phase_top1`, `p_phase_top2`, and `reason_codes` such as 'low\_separation' (if clarity gap is below 0.10), 'low\_confidence' (if pmax is below 0.65), and 'high\_transition\_entropy' (if entropy exceeds 70% of maximum possible entropy).
*   Conditional `new_action_details` are included in the `decision_event` when a new action is performed. This sub-dictionary captures `new_action_id`, `label`, `phase_binding`, `type`, a placeholder `impact_vector_est` (currently `[0.0, 0.0, 0.0, 0.0]`), and `created_from_abst` (indicating if the action was created due to an abstention event).
*   The `learning_event` structure within the `daily_learning_log` is now consistent and correctly integrated. Its `abstention` field accurately reflects the model's clarity state (`event['abstained']`), and all required sub-fields (`new_soft_variable`, `action_taken`, `hard_metrics_snapshot`) are consistently populated with current event data.

### Insights or Next Steps

*   The enhanced logging structure provides a comprehensive dataset for analyzing model behavior, particularly regarding decision-making under uncertainty, user interventions, and the introduction of new actions or soft variables. This will be crucial for model evaluation and future improvements.
*   The current `impact_vector_est` and `outcome_stub` fields are placeholders. The next logical step is to implement the actual mechanisms for calculating these values to provide richer context for learning and analysis.
